# NER Submission — Single Models (Run 2 + Run 3)

Creates two additional submission folders from the prediction files already generated
by `bert_NER_inference_TEST_two_models.ipynb`. **No GPU needed, no inference.**

| Run | Model | Pred file |
|-----|-------|-----------|
| R2 | NB1b-PubMedBERT | `pred_NB1b_pubmed_gold_silver_bronze_TEST.json` |
| R3 | NB1b-BioBERT    | `pred_NB1b_gold_silver_bronze_TEST.json` |

Dev results for reference:
- NB1b_pubmed: Macro-F1=0.7973, Micro-F1=0.8270
- NB1b (BioBERT): Macro-F1=0.7749, Micro-F1=0.8194

## 0. Imports & paths

In [1]:
import json
import copy
from pathlib import Path
from collections import Counter

def find_repo_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / 'data').exists() and (p / 'src').exists():
            return p
    raise FileNotFoundError('Cannot find repo root')

PROJECT_ROOT = find_repo_root(Path.cwd())
PRED_DIR     = PROJECT_ROOT / 'src' / 'ner' / 'predictions'/ 'test_set'

# ⚠️ Set your team ID here
TEAM_ID = "SMTE"   # ← your CLEF 2026 team ID
TASK_ID = "T611"

# Run definitions
RUNS = [
    {
        "run_id":      "R2",
        "system_desc": "NERpubmedbert",
        "pred_file":   PRED_DIR / "pred_NB1b_pubmed_gold_silver_bronze_TEST.json",
        "model_name":  "PubMedBERT (microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract)",
        "backbone":    "PubMedBERT",
        "dev_macro":   "0.7973",
        "dev_micro":   "0.8270",
    },
    {
        "run_id":      "R3",
        "system_desc": "NERbiobert",
        "pred_file":   PRED_DIR / "pred_NB1b_gold_silver_bronze_TEST.json",
        "model_name":  "BioBERT (dmis-lab/biobert-v1.1)",
        "backbone":    "BioBERT",
        "dev_macro":   "0.7749",
        "dev_micro":   "0.8194",
    },
]

print('Project root:', PROJECT_ROOT)
for r in RUNS:
    status = '✓' if r['pred_file'].exists() else '✗ MISSING'
    print(f"  {status}  {r['run_id']}  {r['pred_file'].name}")

Project root: C:\Users\super\Documents\UniPd\ATA\GutBrainIE
  ✓  R2  pred_NB1b_pubmed_gold_silver_bronze_TEST.json
  ✓  R3  pred_NB1b_gold_silver_bronze_TEST.json


## 1. Official dedup + overlap removal

In [2]:
def remove_duplicated_entities(predictions: dict) -> None:
    removed = 0
    for pmid in list(predictions):
        seen, deduped = set(), []
        for e in predictions[pmid]['entities']:
            k = (e['start_idx'], e['end_idx'], e['location'])
            if k not in seen: seen.add(k); deduped.append(e)
            else: removed += 1
        predictions[pmid]['entities'] = deduped
    if removed > 0:
        print(f'  Removed {removed} duplicated entities')


def remove_overlapping_entities(predictions: dict) -> None:
    removed = 0
    for pmid in list(predictions):
        orig = len(predictions[pmid]['entities'])
        groups = {'title': [], 'abstract': []}
        for e in predictions[pmid]['entities']:
            groups[e['location']].append(e)
        keepers = set()
        for loc in groups:
            group = sorted(groups[loc], key=lambda e: e['start_idx'])
            clusters, cluster, cur_end = [], [], None
            for e in group:
                if not cluster:
                    cluster = [e]; cur_end = e['end_idx']
                elif e['start_idx'] < cur_end:
                    cluster.append(e)
                    if e['end_idx'] > cur_end: cur_end = e['end_idx']
                else:
                    clusters.append(cluster); cluster = [e]; cur_end = e['end_idx']
            if cluster: clusters.append(cluster)
            for cl in clusters:
                longest = cl[0]
                for e in cl[1:]:
                    if e['end_idx'] - e['start_idx'] > longest['end_idx'] - longest['start_idx']:
                        longest = e
                keepers.add((longest['start_idx'], longest['end_idx'], longest['location']))
        deduped = []
        for e in predictions[pmid]['entities']:
            k = (e['start_idx'], e['end_idx'], e['location'])
            if k in keepers: deduped.append(e); keepers.discard(k)
        predictions[pmid]['entities'] = deduped
        removed += orig - len(deduped)
    if removed > 0:
        print(f'  Removed {removed} overlapping entities')


print('Dedup/overlap functions ready.')

Dedup/overlap functions ready.


## 2. Submission format validation

In [3]:
LEGAL_ENTITY_LABELS = {
    'anatomical location', 'animal', 'bacteria', 'biomedical technique',
    'chemical', 'DDF', 'dietary supplement', 'drug', 'food', 'gene',
    'human', 'microbiome', 'statistical technique'
}
REQUIRED_FIELDS = {'start_idx', 'end_idx', 'location', 'text_span', 'label'}
FORBIDDEN_FIELDS = {'uri', 'score'}

def validate(predictions: dict, tag: str) -> bool:
    errors = []
    for pmid, obj in predictions.items():
        if set(obj.keys()) != {'entities'}:
            errors.append(f"{pmid}: unexpected top-level keys {set(obj.keys())}")
        for i, e in enumerate(obj['entities']):
            missing = REQUIRED_FIELDS - set(e.keys())
            if missing: errors.append(f"{pmid}[{i}]: missing {missing}")
            forbidden = FORBIDDEN_FIELDS & set(e.keys())
            if forbidden: errors.append(f"{pmid}[{i}]: forbidden {forbidden}")
            if e.get('label') not in LEGAL_ENTITY_LABELS:
                errors.append(f"{pmid}[{i}]: illegal label '{e.get('label')}'")
            if e.get('location') not in {'title', 'abstract'}:
                errors.append(f"{pmid}[{i}]: illegal location '{e.get('location')}'")
    if errors:
        print(f'  ⚠️  {tag}: {len(errors)} error(s)')
        for err in errors[:10]: print(f'    {err}')
        return False
    print(f'  ✓ {tag}: validation passed')
    return True

print('Validation function ready.')

Validation function ready.


## 3. Build submission folders

In [4]:
for run in RUNS:
    folder_name = f"{TEAM_ID}_{TASK_ID}_{run['run_id']}_{run['system_desc']}"
    print(f"\n{'='*55}")
    print(f"  {folder_name}")
    print(f"{'='*55}")

    # Load predictions
    with run['pred_file'].open(encoding='utf-8') as f:
        preds = json.load(f)
    print(f"  Loaded {len(preds)} articles, "
          f"{sum(len(v['entities']) for v in preds.values())} entities")

    # Apply official dedup/overlap
    remove_duplicated_entities(preds)
    remove_overlapping_entities(preds)

    # Strip any forbidden fields (score, uri) just in case
    for obj in preds.values():
        for e in obj['entities']:
            e.pop('score', None)
            e.pop('uri', None)

    # Validate
    valid = validate(preds, folder_name)
    if not valid:
        print('  ✗ Skipping save due to validation errors.')
        continue

    total = sum(len(v['entities']) for v in preds.values())
    label_counts = Counter(
        e['label']
        for obj in preds.values()
        for e in obj['entities']
    )
    print(f"  Total entities after cleanup: {total}")

    # Create folder
    out_dir = PRED_DIR / folder_name
    out_dir.mkdir(parents=True, exist_ok=True)

    # Save JSON
    json_path = out_dir / f"{folder_name}.json"
    with json_path.open('w', encoding='utf-8') as f:
        json.dump(preds, f, ensure_ascii=False, indent=2)
    print(f"  ✓ Saved JSON : {json_path.name}")

    # Write .meta
    meta_content = (
        f"Team ID:         {TEAM_ID}\n"
        f"Task ID:         {TASK_ID}\n"
        f"Run ID:          {run['run_id']}\n"
        "\n"
        "Type of training:\n"
        f"  Fine-tuning of {run['backbone']} on GutBrainIE 2026 training data\n"
        "  with token classification (BIO tagging, 27 labels).\n"
        "\n"
        "Pre-processing methods:\n"
        "  - Separate inference on title and abstract segments to preserve character offsets.\n"
        "  - BIO repair for malformed I-tags at sequence start.\n"
        "  - Two-pass thresholding: high-precision pass + recall-boost pass for\n"
        "    (chemical, bacteria, food, dietary supplement) labels.\n"
        "  - Label-aware post-processing: gene/chemical remap, food/dietary supplement remap,\n"
        "    FP filtering based on span text.\n"
        "  - Span score = median token probability.\n"
        "  - Deduplication and soft overlap pruning.\n"
        "  - Final dedup and overlap removal following the official evaluate.py logic.\n"
        "\n"
        "Training data used:\n"
        "  Gold + Silver + Bronze annotations from GutBrainIE 2026 training set.\n"
        "  Priority merge: when same PMID appears in multiple quality levels,\n"
        "  higher-quality annotation is kept.\n"
        "\n"
        "Relevant details of the run:\n"
        f"  Single model: {run['model_name']}\n"
        "  Fine-tuned on Gold+Silver+Bronze with class-weighted cross-entropy,\n"
        "  lr=3e-5, 5 epochs, batch size 8 (grad accum 2 steps).\n"
        "  No ensemble — single model inference only.\n"
        f"  Dev set results: Macro-F1={run['dev_macro']}, Micro-F1={run['dev_micro']}.\n"
        "\n"
        "GitHub repository:\n"
        "  https://github.com/sofiamaule/GutBrainIE.git\n"
    )
    meta_path = out_dir / f"{folder_name}.meta"
    meta_path.write_text(meta_content, encoding='utf-8')
    print(f"  ✓ Saved META : {meta_path.name}")

print("\n✓ Done. Both single-model submission folders created.")


  SMTE_T611_R2_NERpubmedbert
  Loaded 80 articles, 2422 entities
  ✓ SMTE_T611_R2_NERpubmedbert: validation passed
  Total entities after cleanup: 2422
  ✓ Saved JSON : SMTE_T611_R2_NERpubmedbert.json
  ✓ Saved META : SMTE_T611_R2_NERpubmedbert.meta

  SMTE_T611_R3_NERbiobert
  Loaded 80 articles, 2554 entities
  ✓ SMTE_T611_R3_NERbiobert: validation passed
  Total entities after cleanup: 2554
  ✓ Saved JSON : SMTE_T611_R3_NERbiobert.json
  ✓ Saved META : SMTE_T611_R3_NERbiobert.meta

✓ Done. Both single-model submission folders created.


## 4. Summary stats for all three runs

In [5]:
# Compare entity counts across the 3 runs
run_configs = [
    ("R1 Ensemble",    PRED_DIR / "SMTE_T611_R1_NERensemble"    / "SMTE_T611_R1_NERensemble.json"),
    ("R2 PubMedBERT",  PRED_DIR / "SMTE_T611_R2_NERpubmedbert" / "SMTE_T611_R2_NERpubmedbert.json"),
    ("R3 BioBERT",     PRED_DIR / "SMTE_T611_R3_NERbiobert"    / "SMTE_T611_R3_NERbiobert.json"),
]

print(f"{'Run':<20s}  {'Articles':>8}  {'Entities':>9}")
print("-" * 42)
for name, path in run_configs:
    if not path.exists():
        print(f"{name:<20s}  {'FILE NOT FOUND':>20}")
        continue
    with path.open(encoding='utf-8') as f:
        data = json.load(f)
    n_ents = sum(len(v['entities']) for v in data.values())
    print(f"{name:<20s}  {len(data):>8}  {n_ents:>9}")

Run                   Articles   Entities
------------------------------------------
R1 Ensemble                 80       2639
R2 PubMedBERT               80       2422
R3 BioBERT                  80       2554


In [6]:
import zipfile
from pathlib import Path

PRED_DIR   = PROJECT_ROOT / 'src' / 'ner' / 'predictions' /'test_set'
TEAM_ID    = "SMTE"   # ← stesso di prima
ZIP_NAME   = f"{TEAM_ID}_GutBrainIE_2026.zip"
ZIP_PATH   = PRED_DIR / ZIP_NAME

RUN_FOLDERS = [
    "SMTE_T611_R1_NERensemble",
    "SMTE_T611_R2_NERpubmedbert",
    "SMTE_T611_R3_NERbiobert",
]

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for folder_name in RUN_FOLDERS:
        folder_path = PRED_DIR / folder_name
        for file in folder_path.iterdir():
            # arcname = path inside the zip: folder/file
            arcname = f"{folder_name}/{file.name}"
            zf.write(file, arcname=arcname)
            print(f"  Added: {arcname}")

print(f"\n✓ Created: {ZIP_PATH}")
print(f"  Size: {ZIP_PATH.stat().st_size / 1024:.1f} KB")

  Added: SMTE_T611_R1_NERensemble/SMTE_T611_R1_NERensemble.json
  Added: SMTE_T611_R1_NERensemble/SMTE_T611_R1_NERensemble.meta
  Added: SMTE_T611_R2_NERpubmedbert/SMTE_T611_R2_NERpubmedbert.json
  Added: SMTE_T611_R2_NERpubmedbert/SMTE_T611_R2_NERpubmedbert.meta
  Added: SMTE_T611_R3_NERbiobert/SMTE_T611_R3_NERbiobert.json
  Added: SMTE_T611_R3_NERbiobert/SMTE_T611_R3_NERbiobert.meta

✓ Created: C:\Users\super\Documents\UniPd\ATA\GutBrainIE\src\ner\predictions\test_set\SMTE_GutBrainIE_2026.zip
  Size: 118.6 KB
